# Old PPO vs PPO-v2
Fixed paired deck/seat evaluation. No training and no submission.

In [ ]:
import glob, os, sys, json, multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
import numpy as np
for package in ('src','cg'):
    matches = glob.glob(f'/kaggle/input/**/{package}/__init__.py', recursive=True)
    root = os.path.dirname(os.path.dirname(matches[0]))
    sys.path.insert(0, root)
deck_match = glob.glob('/kaggle/input/**/deck_lucario.csv', recursive=True)[0]
config_dir = os.path.dirname(deck_match)
weight_files = glob.glob('/kaggle/input/**/old_ppo.npz', recursive=True) + glob.glob('/kaggle/input/**/ppo_v2.npz', recursive=True)
if len(weight_files) != 2: raise RuntimeError(f'weights={weight_files}')
old_path = [p for p in weight_files if p.endswith('old_ppo.npz')][0]
new_path = [p for p in weight_files if p.endswith('ppo_v2.npz')][0]
from src.selfplay import load_decks, make_eval_schedule, tally_eval_results
from src.policy_numpy import load_params
decks = load_decks(config_dir, ['Lucario','Crustle','Alakazam'])
old = load_params(old_path); new = load_params(new_path)
print('loaded', old_path, new_path)

In [ ]:
from src.rollout_worker import init_worker, play_game
def run_side(active, opponent, tasks):
    with ProcessPoolExecutor(max_workers=4, mp_context=mp.get_context('spawn'), initializer=init_worker, initargs=(active, [opponent], 64, 512, 0.0, 0.99)) as ex:
        return list(ex.map(play_game, tasks))
names = list(decks)
tasks = make_eval_schedule(names, decks, 250, 424242)
for task in tasks: task['record_transitions'] = False
paired_tasks = [dict(task, active_seat=1-task['active_seat']) for task in tasks]
# Both policies play the same deck/seed schedule, with seats reversed.
old_results = run_side(old, new, tasks)
new_results = run_side(new, old, paired_tasks)
def count(results):
    wins=losses=draws=errors=0
    for r in results:
        if r.get('error'): errors += 1
        elif r['winner'] < 0: draws += 1
        elif r['winner'] == r['active_seat']: wins += 1
        else: losses += 1
    return {'wins':wins,'losses':losses,'draws':draws,'errors':errors,'n':wins+losses+draws,'win_rate':wins/max(1,wins+losses+draws)}
old_score=count(old_results); new_score=count(new_results)
print(json.dumps({'old_ppo':old_score,'ppo_v2':new_score}, indent=2))
with open('/kaggle/working/head2head.json','w') as f: json.dump({'old_ppo':old_score,'ppo_v2':new_score},f,indent=2)